# VLM extraction pipeline — exact prompt walkthrough

**Goal:** for one image and one feature, show *exactly* what we send to the VLM, what comes back, and how the multi-step pipeline composes.

Use this as a reference when revisiting the VLM phase or showcasing how feature extraction works.

---

## Pipeline overview

```
┌──────────────────────┐
│ 0. Visibility gate   │  pre-screen: is this region visible at all?
│    (DB-side, prior)  │  source: vlm_graded annotations from a separate VLM pass
└──────────┬───────────┘
           │ pass
           ▼
┌──────────────────────┐
│ 1a. Path A           │  single-shot: system + (optional few-shot) + user → structured output
│    (default for      │  used by cap_color, volva_presence, ring_presence, substrate, ...
│    most features)    │
└──────────┬───────────┘
           OR
┌──────────────────────┐
│ 1b. Path C (staged)  │  two calls:
│    (hymenium_type,   │    stage 1 → coarse family
│    stem_shape only)  │    stage 2 → leaf class within family (dispatched on stage 1)
└──────────────────────┘
```

Pick a feature + image at the top, then run all cells.

## Configuration

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# ── Edit these to walk through any feature/image combination ──────────────
FEATURE = "hymenium_type"           # one of: hymenium_type, cap_color, volva_presence,
                                    # ring_presence, substrate, surface_texture,
                                    # stem_shape, cap_shape
IMAGE_ID: str | None = None         # set to a 12-char image_id, or leave None to auto-pick
MODEL = "gemma4:31b-cloud"          # production model used during the eval phase
USE_FEW_SHOT = True                 # Path A: include reference images if available
# ─────────────────────────────────────────────────────────────────────────

PROCESSED_DIR = PROJECT_ROOT / "data" / "images" / "processed" / "v1"
PROCESSED_896 = PROJECT_ROOT / "data" / "images" / "processed" / "vlm_896_cc"
REFERENCE_DIR = PROJECT_ROOT / "vision" / "labeling" / "reference_images"

print(f"feature: {FEATURE}")
print(f"model:   {MODEL}")
print(f"project: {PROJECT_ROOT}")

## Step 1 — pick a test image and see ground truth

If `IMAGE_ID` is `None`, we pick a recent eval image for the chosen feature. Then we display the image and its species-level GT.

In [ ]:
import pandas as pd
from IPython.display import Image, display
from sqlalchemy import text

from db.connection import get_session

# Auto-pick: any image with both a species_propagated annotation for FEATURE
# and a processed file on disk. Robust across features (eval-dir naming
# conventions are inconsistent).
if IMAGE_ID is None:
    with get_session() as session:
        rows = session.execute(
            text(
                "SELECT a.image_id, r.species, a.feature_value "
                "FROM image_annotations a "
                "JOIN image_registry r ON a.image_id = r.image_id "
                "WHERE a.feature_name = :fn "
                "  AND a.annotation_type = 'species_propagated' "
                "  AND a.feature_value IS NOT NULL "
                "ORDER BY a.image_id LIMIT 50"
            ),
            {"fn": FEATURE},
        ).fetchall()
    for iid, sp_, gt_ in rows:
        if (PROCESSED_DIR / f"{iid}.jpg").exists():
            IMAGE_ID = iid
            break
    if IMAGE_ID is None:
        raise SystemExit(
            f"No image with both an annotation for {FEATURE!r} and a processed file. "
            "Set IMAGE_ID manually."
        )

image_path = PROCESSED_DIR / f"{IMAGE_ID}.jpg"
image_path_896 = PROCESSED_896 / f"{IMAGE_ID}.jpg"
INFERENCE_PATH = image_path_896 if image_path_896.exists() else image_path

with get_session() as session:
    sp = session.execute(
        text("SELECT species FROM image_registry WHERE image_id = :iid"),
        {"iid": IMAGE_ID},
    ).scalar()
    gt_row = session.execute(
        text(
            "SELECT feature_value FROM image_annotations "
            "WHERE image_id=:iid AND feature_name=:fn AND annotation_type='species_propagated'"
        ),
        {"iid": IMAGE_ID, "fn": FEATURE},
    ).scalar()

print(f"image_id:        {IMAGE_ID}")
print(f"species:         {sp}")
print(f"GT ({FEATURE}): {gt_row}")
print(f"file (display):  {image_path}")
print(f"file (inference) {INFERENCE_PATH}    {'(896×896 cc)' if INFERENCE_PATH==image_path_896 else '(v1 raw)'}")
display(Image(filename=str(INFERENCE_PATH), width=400))

## Step 2 — visibility gate (pre-screen, DB-side, NOT a chain-of-questions)

Before any feature extraction runs, we check whether the relevant region is even visible. This is a separate VLM pass (`vlm_graded`) that ran once over the whole image pool — it is **not** a chain step in the per-feature extraction.

`_VISIBILITY_REQUIREMENTS` maps each feature to the gate field it depends on:

In [ ]:
from vision.data.manifest import _VISIBILITY_REQUIREMENTS

gate_field = _VISIBILITY_REQUIREMENTS.get(FEATURE)
print(f"_VISIBILITY_REQUIREMENTS[{FEATURE!r}] = {gate_field!r}")

if gate_field is None:
    print("  → no visibility gate (every image passes)")
else:
    with get_session() as session:
        gate_val = session.execute(
            text(
                "SELECT feature_value FROM image_annotations "
                "WHERE image_id=:iid AND feature_name=:gate AND annotation_type='vlm_graded'"
            ),
            {"iid": IMAGE_ID, "gate": gate_field},
        ).scalar()
    print(f"  vlm_graded[{gate_field}] = {gate_val!r}")
    print(
        "  → image PASSES gate" if gate_val in (None, "true")
        else "  → image WOULD BE EXCLUDED from training/eval manifest"
    )

## Step 3 — Path A: single-shot extraction (with optional few-shot anchors)

**Path A is the project default.** Three components:
1. **System prompt** — role + confidence scale
2. **(Optional) few-shot block** — interleaved `text + image` blocks showing one reference photo per class. Only built if `vision/labeling/reference_images/<feature>/` contains `.jpg` files.
3. **User prompt** — describe-then-classify ordering, rare-first class enumeration, anti-trap guidance, MCQA tail.

The full structured-output schema (e.g. `HymeniumTypeResult`) enforces field ordering: `visible → visual_description → reasoning → classification → confidence`.

In [ ]:
from vision.labeling.vlm_feature_prompts import PROMPT_REGISTRY
from vision.labeling.vlm_feature_schemas import FEATURE_REGISTRY

schema_cls = FEATURE_REGISTRY[FEATURE]["schema"]
system_prompt, user_prompt = PROMPT_REGISTRY[FEATURE]

print("=" * 70)
print(f"SYSTEM PROMPT for {FEATURE}")
print("=" * 70)
print(system_prompt)
print()
print("=" * 70)
print(f"USER PROMPT for {FEATURE}")
print("=" * 70)
print(user_prompt)
print()
print("=" * 70)
print(f"STRUCTURED OUTPUT SCHEMA: {schema_cls.__name__}")
print("=" * 70)
for fname, finfo in schema_cls.model_fields.items():
    print(f"  {fname:22} {finfo.annotation}")

In [ ]:
# Few-shot anchors: do reference images exist for this feature?
ref_dir = REFERENCE_DIR / FEATURE
ref_images = sorted(ref_dir.glob("*.jpg")) if ref_dir.exists() else []

print(f"reference dir:    {ref_dir}")
print(f"few-shot enabled: {USE_FEW_SHOT and bool(ref_images)}")
print(f"reference images: {len(ref_images)}")

if USE_FEW_SHOT and ref_images:
    print("\nThe message payload becomes interleaved text + image blocks:")
    print("  [text: preamble]")
    for ri in ref_images:
        label = ri.stem.replace("_ref", "").upper()
        print(f"  [text: {label!r}] [image: {ri.name}]")
    print("  [text: full user prompt + 'Now classify THIS mushroom:']")
    print("  [image: query image]")
    print("\nReference images shown to the VLM:")
    for ri in ref_images:
        print(f"\n— {ri.stem.replace('_ref', '').upper()}")
        display(Image(filename=str(ri), width=200))
else:
    print("\nNo few-shot block — the message is just system + user-prompt + query image.")

In [ ]:
# Run the actual Path A extraction. This makes ONE live call to the cloud VLM.
from vision.labeling.vlm_feature_extractor import extract_feature

print(f"calling {MODEL} for {FEATURE}...")
path_a_result = extract_feature(
    image_path=INFERENCE_PATH,
    feature_name=FEATURE,
    model_name=MODEL,
    use_few_shot=USE_FEW_SHOT,
)

print("\n" + "=" * 70)
print("PATH A — STRUCTURED RESPONSE")
print("=" * 70)
for fname in schema_cls.model_fields:
    print(f"  {fname:22} {getattr(path_a_result, fname)!r}")

## Step 4 — Path C: staged chain-of-questions (only if registered)

Path C is the *only* place we do an actual chain of VLM calls per image:

1. **Stage 1** — coarse family classification (e.g., for `hymenium_type`: `linear_radial | punctate | featureless_or_internal`)
2. **Stage 2** — within-family disambiguation, dispatched on stage 1's answer (e.g., `linear_radial` → `gills | ridges`)
3. **Min-confidence merge** — final confidence is the lower of the two stages

Only registered features (`hymenium_type`, `stem_shape`) have a staged config. Path C only helps when stage 1 is genuinely separable; on threshold problems it doesn't help (lesson from `stem_shape`).

In [ ]:
from vision.labeling.vlm_staged_schemas import STAGED_REGISTRY

if FEATURE not in STAGED_REGISTRY:
    print(f"{FEATURE!r} has no staged config — skipping Path C.")
    print(f"Staged features available: {sorted(STAGED_REGISTRY)}")
else:
    cfg = STAGED_REGISTRY[FEATURE]
    print("=" * 70)
    print(f"STAGE 1 SYSTEM PROMPT — {FEATURE} (coarse family)")
    print("=" * 70)
    print(cfg.stage1_system)
    print()
    print("=" * 70)
    print(f"STAGE 1 USER PROMPT — {FEATURE}")
    print("=" * 70)
    print(cfg.stage1_user)
    print()
    print(f"Stage 1 output schema: {cfg.stage1_schema.__name__}")
    print(f"Family field:          {cfg.family_field!r}")
    print(f"Allowed families:      {sorted(cfg.families)}")

In [ ]:
# Run stage 1 live (only if Path C is registered for this feature).
from llm.client import structured_vision_completion

if FEATURE not in STAGED_REGISTRY:
    print("skipping Path C")
    stage1 = None
else:
    cfg = STAGED_REGISTRY[FEATURE]
    print(f"calling {MODEL} for stage 1...")
    stage1 = structured_vision_completion(
        prompt=cfg.stage1_user,
        image_paths=[INFERENCE_PATH],
        response_model=cfg.stage1_schema,
        system=cfg.stage1_system,
        model=MODEL,
    )
    print("\n" + "=" * 70)
    print(f"STAGE 1 RESPONSE — {cfg.stage1_schema.__name__}")
    print("=" * 70)
    for fname in cfg.stage1_schema.model_fields:
        print(f"  {fname:22} {getattr(stage1, fname)!r}")

In [ ]:
# Stage 2: dispatch on the family stage 1 chose, then run the matching prompt.
if stage1 is None:
    print("no stage 1 — skipping stage 2")
    stage2 = None
    final_staged = None
else:
    cfg = STAGED_REGISTRY[FEATURE]
    family = getattr(stage1, cfg.family_field)
    print(f"stage 1 picked family = {family!r}")

    if family is None or stage1.confidence == "cannot_tell":
        print("\nstage 1 abstained → no stage-2 call; final = null with cannot_tell")
        stage2 = None
        final_staged = cfg.final_schema(
            visible=bool(stage1.visible),
            visual_description=stage1.visual_description,
            reasoning=stage1.reasoning,
            confidence="cannot_tell",
            **{cfg.leaf_field: None},
        )
    else:
        fam_cfg = cfg.families[family]
        print("\n" + "=" * 70)
        print(f"STAGE 2 SYSTEM PROMPT — family={family!r}")
        print("=" * 70)
        print(fam_cfg.system)
        print()
        print("=" * 70)
        print(f"STAGE 2 USER PROMPT — family={family!r}")
        print("=" * 70)
        print(fam_cfg.user)
        print()
        print(f"calling {MODEL} for stage 2 ({family})...")
        stage2 = structured_vision_completion(
            prompt=fam_cfg.user,
            image_paths=[INFERENCE_PATH],
            response_model=fam_cfg.schema,
            system=fam_cfg.system,
            model=MODEL,
        )
        print("\n" + "=" * 70)
        print(f"STAGE 2 RESPONSE — {fam_cfg.schema.__name__}")
        print("=" * 70)
        for fname in fam_cfg.schema.model_fields:
            print(f"  {fname:22} {getattr(stage2, fname)!r}")

        # Min-confidence merge (same logic as extract_feature_staged)
        rank = {"cannot_tell": 0, "low": 1, "high": 2}
        merged_conf = min(stage1.confidence, stage2.confidence, key=rank.get)
        leaf_value = getattr(stage2, cfg.leaf_field)
        if leaf_value is None or stage2.confidence == "cannot_tell":
            merged_conf = "cannot_tell"
            leaf_value = None
        final_staged = cfg.final_schema(
            visible=True,
            visual_description=stage2.visual_description or stage1.visual_description,
            reasoning=stage2.reasoning or stage1.reasoning,
            confidence=merged_conf,
            **{cfg.leaf_field: leaf_value},
        )
        print("\n" + "=" * 70)
        print("PATH C — FINAL MERGED RESULT")
        print("=" * 70)
        for fname in cfg.final_schema.model_fields:
            print(f"  {fname:22} {getattr(final_staged, fname)!r}")

## Step 5 — side by side: GT vs Path A vs Path C

In [ ]:
leaf_field = (
    STAGED_REGISTRY[FEATURE].leaf_field
    if FEATURE in STAGED_REGISTRY
    else FEATURE_REGISTRY[FEATURE]["field"]
)

rows = [
    ("species GT",  gt_row,                                  "—"),
    ("Path A",      getattr(path_a_result, leaf_field),      path_a_result.confidence),
]
if final_staged is not None:
    rows.append(("Path C",  getattr(final_staged, leaf_field), final_staged.confidence))

print(f"{'source':12} {'value':22} {'confidence':12}")
print("-" * 50)
for src, val, conf in rows:
    print(f"{src:12} {str(val):22} {conf!s:12}")

## Reading the results

**Why this notebook helps you reason about prompt engineering vs visual anchors:**

- The `visual_description` and `reasoning` fields show what the VLM *thought it saw*. If those are accurate but the final class is wrong, the failure is at the classification step (anti-trap text or class-boundary logic). If `visual_description` itself is wrong, no amount of prompt engineering will recover it — the model isn't seeing the relevant detail at 896×896.
- Compare Path A (no anchors) vs Path A with anchors by toggling `USE_FEW_SHOT`. Anchors help most when class-boundaries are visually subtle (e.g., gills vs ridges in `hymenium_type`); they help less when the bottleneck is regional visibility (e.g., `surface_texture` at 896×896).
- Path C only beats Path A when stage 1 splits classes that look genuinely different (hymenium texture families). It does *not* help when the dominant failure is a fuzzy threshold within one visual family (`stem_shape` equal-vs-widening).

**To explore:**
- Change `FEATURE` to `cap_color` (Path A + few-shot only) or `volva_presence` (Path A only).
- Change `IMAGE_ID` to a different image to compare answers across the same feature.
- Toggle `USE_FEW_SHOT = False` to see the anchor-free response on the same image.